# 🏠 Week 2 Homework 2 of 2 · The API grand tour

**First move: File → Save a copy in Drive, then work in YOUR copy.**

Here is the industry secret that saves you months: most LLM providers copied the same API shape, the **chat-completions** format, and **OpenRouter** puts one door in front of all of them. One key, one address, and the model becomes just a STRING: swap the string, you have swapped from Meta's kitchen to Google's to OpenAI's to Anthropic's. This homework: one code shape, many models.

**What is free and what is not, honestly:**

| Model (via OpenRouter) | Cost | You get |
|---|---|---|
| Models ending in `:free` | Free (daily cap, no credits needed) | A rotating set of strong open models: our lane |
| `openai/gpt-...-mini` tiers | Paid: $5 credits last a long time | GPT models |
| `anthropic/claude-...` | Paid | Claude models |
| `google/gemini-...` | Cheap paid tiers | Gemini models |

Do everything on the free lane first. Add credits only if you want the closed models; every task below works with just the `:free` pair.

## 🔑 Step 0 · One key, step by step

Your OpenRouter key from class and HW1 is the ONLY key this notebook needs. No key yet: [openrouter.ai](https://openrouter.ai) → sign in → Keys → Create Key (`sk-or-...`, shown once).

**Optional, for the closed models:** on OpenRouter, avatar → **Credits** → add **$5** (one-time, no subscription). That unlocks `openai/...`, `anthropic/...` and every other paid model through the SAME key. Skip it tonight and stay on the `:free` lane; nothing below requires paying.

In [1]:
"""
🔑 KEY SETUP · run me first, every session. ONE key, three doors:
Colab Secrets -> local .env file -> paste at an invisible prompt.
"""

!pip install -q openai

import os


def load_key(name):
    """Colab Secrets -> .env -> ask."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            print(name, "loaded from Colab Secrets.")
            return value
    except Exception:
        pass
    try:
        with open(".env") as f:
            for line in f.read().splitlines():
                if line.startswith(name + "="):
                    value = line.split("=", 1)[1].strip()
                    if value:
                        print(name, "loaded from the local .env file.")
                        return value
    except FileNotFoundError:
        pass
    from getpass import getpass
    return getpass(name + ": ")


os.environ["OPENROUTER_API_KEY"] = load_key("OPENROUTER_API_KEY")

print("Key loaded. One key, many kitchens. Never in code, never in git.")

Key loaded. One key, many kitchens. Never in code, never in git.


## 📖 Worked example 1 · The shape

The chat-completions format, the one shape to learn. Three parts:

1. **The client**: made with a key and a `base_url`, the address of the door. For us the door is always OpenRouter; the same `openai` package would reach any provider's own door by changing this one line. That is the whole trick.
2. **The messages list**: a conversation. Each entry has a `role`: `"system"` is the standing instruction that shapes every answer (persona, tone, rules), `"user"` is you. Dicts in a list: Week 1 knowledge, new outfit.
3. **The answer**: at `response.choices[0].message.content`, with the token receipt at `response.usage`.

Run it. Note the model string's shape: provider slash model-name, plus `:free` for the no-credit lane.

In [2]:
# WORKED EXAMPLE 1 · full solution. Run me.

from openai import OpenAI

client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",   # the one door, always
)

response = client.chat.completions.create(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    messages=[
        {"role": "system", "content": "You are a concise assistant. Two sentences maximum, always."},
        {"role": "user", "content": "What is OpenRouter and why does one key reach many models?"},
    ],
)

print(response.choices[0].message.content)
print("\ntokens:", response.usage.prompt_tokens, "in,",
      response.usage.completion_tokens, "out")

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1791072000000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None}}, 'user_id': 'user_3IaiK4Tum45bNfOM07aDWCGJLF7'}

## 📖 Worked example 2 · Same shape, every kitchen

Now the payoff: one client, one `ask_model` helper, and models become entries in a dict. Two `:free` defaults for the no-credit lane, two closed models for anyone who added credits (flip the flag). If a `:free` default ever 404s (the free set rotates), run the discovery loop at the bottom of the cell and swap in a current one: model names are configuration, not code.

In [ ]:
# WORKED EXAMPLE 2 · full solution. Run me.

from openai import OpenAI

client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

HAS_CREDITS = False   # flip to True if you added $5 on OpenRouter

MODELS = {
    "nemotron-120b": {"id": "nvidia/nemotron-3-super-120b-a12b:free", "free": True},
    "nex-pro": {"id": "qwen/qwen3.8-27b:free", "free": True},
    "gpt-mini": {"id": "openai/gpt-4o-mini", "free": False},
    "claude-haiku": {"id": "anthropic/claude-haiku-4.5", "free": False},
}


def available_models():
    """The models your account can call right now."""
    ready = []
    for name in MODELS:
        if MODELS[name]["free"] or HAS_CREDITS:
            ready.append(name)
    return ready


def ask_model(name, prompt, system="You are a helpful, concise assistant."):
    """One prompt to one model, chat-completions shape."""
    response = client.chat.completions.create(
        model=MODELS[name]["id"],
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content.strip()


print("Models ready:", available_models())
for name in available_models():
    print("\n---", name.upper(), "says ---")
    print(ask_model(name, "In one sentence: what is special about you as a model?"))

# A default 404s? The :free set rotates. Discover today's free models:
# for m in client.models.list():
#     if m.id.endswith(":free"):
#         print(m.id)

## ✏️ Your turn · four practice tasks

All four tasks work on the free pair of models. Credits give you more contestants, not more homework.

In [ ]:
"""
✏️ TASK 1 · The shootout
Ask every available model the SAME question and collect the answers
into a dict: {model_name: answer_text}. Then read them side by side.
"""

question = "Explain what an API key is, in exactly two sentences, to a beginner."

answers = {}
# TODO: loop over available_models(), fill the dict with ask_model
for name in available_models():
    answers[name] = ask_model(name, question)


assert len(answers) >= 2, "Need at least the two free models here."
for name in answers:
    assert isinstance(answers[name], str) and len(answers[name]) > 20
    print("---", name.upper(), "---")
    print(answers[name])
    print()
print("✅ Task 1 passed. Same question, different models: spot the personality differences.")

In [ ]:
"""
✏️ TASK 2 · JSON extraction on open Llama
HW1's invoice trick, now through open Llama. Build parse_with(model_key, text)
that asks for ONLY JSON with keys vendor, total_lkr, city, unwraps
``` fences, and returns the dict (or {"error": "unparseable"}).
"""

import json

invoice = """LANKA APPLIANCES (PVT) LTD, 214 Galle Road, Colombo 03.
Invoice INV-2291. 1x blender LKR 9,500, 1x kettle LKR 3,250.
TOTAL: LKR 12,750. Payment due within 14 days."""

# TODO: def parse_with(model_key, text): ...
def parse_with(model_key, text):
    """Extract vendor / total_lkr / city from text via any model."""
    reply = ask_model(
        model_key,
        "Extract from this invoice. Reply with ONLY a JSON object with "
        "exactly these keys: vendor (string), total_lkr (number, the "
        "grand total), city (string).\n\n" + text,
    )
    reply = reply.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(reply)
    except json.JSONDecodeError:
        return {"error": "unparseable"}


record = parse_with("nemotron-120b", invoice)
assert "error" not in record, f"Parse failed: {record}"
assert sorted(record.keys()) == ["city", "total_lkr", "vendor"]
assert record["total_lkr"] == 12750
print(record)
print("✅ Task 2 passed: the JSON pattern is model-independent. That is the point.")

In [ ]:
"""
✏️ TASK 3 · The speed race
Different models, different speeds, same door. Measure it. time.time()
gives the clock in seconds; the difference around a call is latency.
"""

import time

# TODO: for each available model, time ONE ask_model call with the
# prompt "Say OK." and append {"model": name, "seconds": elapsed}
# to timings.
timings = []
for name in available_models():
    start = time.time()
    ask_model(name, "Say OK.")
    elapsed = time.time() - start
    timings.append({"model": name, "seconds": elapsed})


assert len(timings) >= 2
for t in timings:
    assert sorted(t.keys()) == ["model", "seconds"]
    print(f"{t['model']:>12}: {t['seconds']:.2f}s")
print("✅ Task 3 passed. Speed differs even through one door. Week 4 makes this a scoreboard column.")

In [ ]:
"""
✏️ TASK 4 · The judge (a Week 11 idea, five weeks early)
Have llama-70b summarise the announcement below in one sentence. Then
ask qwen-72b to grade it: ONLY JSON, {"score": 1-10, "reason": short}.
One model judging another: remember this cell in Week 11.
"""

announcement = """Dear valued customers, from the first of next month our
Wellawatte and Nugegoda branches will open 8am to 9pm on weekdays. Home
delivery now covers Mount Lavinia and Dehiwala, free above LKR 5,000,
otherwise LKR 400."""

# NOTE: the model keys in this notebook are nemotron-120b and nex-pro,
# so nemotron-120b summarises and nex-pro judges.
summary = ask_model("nemotron-120b", "Summarise in exactly one sentence:\n\n" + announcement)
# TODO: verdict = ... (judge, JSON with score + reason; reuse the
#       fence unwrapping from Task 2)
reply = ask_model(
    "nex-pro",
    "Grade this one-sentence summary of the announcement for accuracy "
    "and completeness. Reply with ONLY a JSON object with exactly these "
    "keys: score (integer 1 to 10), reason (one short sentence).\n\n"
    "Announcement:\n" + announcement + "\n\nSummary:\n" + summary,
)
reply = reply.replace("```json", "").replace("```", "").strip()
verdict = json.loads(reply)


assert sorted(verdict.keys()) == ["reason", "score"], f"Got: {verdict}"
assert isinstance(verdict["score"], int) and 1 <= verdict["score"] <= 10
print("Summary:", summary)
print("Verdict:", verdict)
print("✅ Task 4 passed: you just ran your first LLM-as-judge.")

## 🚢 Ship it

Same as HW1: File → Download → Download .ipynb, then repo page → **Add file → Upload files**, rename to `week02/homework2_grand_tour.ipynb`, commit message `Week 2 HW2: one door, N models`, Commit.

**Key hygiene reminder before you ship:** your keys are not in the notebook (getpass made sure), so uploading is safe. If you ever DO spot a key in a cell output, clear outputs before shipping: Edit → Clear all outputs.

## 🔑 Solutions

**Try for 15 minutes per task before you peek.** These need the key cell and Worked Example 2 run first.

In [ ]:
# SOLUTION · Task 1
question = "Explain what an API key is, in exactly two sentences, to a beginner."

answers = {}
for name in available_models():
    answers[name] = ask_model(name, question)

for name in answers:
    print("---", name.upper(), "---")
    print(answers[name])
    print()

In [ ]:
# SOLUTION · Task 2
import json

invoice = """LANKA APPLIANCES (PVT) LTD, 214 Galle Road, Colombo 03.
Invoice INV-2291. 1x blender LKR 9,500, 1x kettle LKR 3,250.
TOTAL: LKR 12,750. Payment due within 14 days."""

def parse_with(model_key, text):
    """Extract vendor / total_lkr / city from text via any model."""
    reply = ask_model(
        model_key,
        "Extract from this invoice. Reply with ONLY a JSON object with "
        "exactly these keys: vendor (string), total_lkr (number, the "
        "grand total), city (string).\n\n" + text,
    )
    reply = reply.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(reply)
    except json.JSONDecodeError:
        return {"error": "unparseable"}

record = parse_with("nemotron-120b", invoice)
print(record)

In [ ]:
# SOLUTION · Task 3
import time

timings = []
for name in available_models():
    start = time.time()
    ask_model(name, "Say OK.")
    elapsed = time.time() - start
    timings.append({"model": name, "seconds": elapsed})

for t in timings:
    print(f"{t['model']:>12}: {t['seconds']:.2f}s")

In [ ]:
# SOLUTION · Task 4
import json

announcement = """Dear valued customers, from the first of next month our
Wellawatte and Nugegoda branches will open 8am to 9pm on weekdays. Home
delivery now covers Mount Lavinia and Dehiwala, free above LKR 5,000,
otherwise LKR 400."""

summary = ask_model("nemotron-120b", "Summarise in exactly one sentence:\n\n" + announcement)

reply = ask_model(
    "nex-pro",
    "Grade this one-sentence summary of the announcement for accuracy "
    "and completeness. Reply with ONLY a JSON object with exactly these "
    'keys: score (integer 1 to 10), reason (one short sentence).\n\n'
    "Announcement:\n" + announcement + "\n\nSummary:\n" + summary,
)
reply = reply.replace("```json", "").replace("```", "").strip()
verdict = json.loads(reply)

print("Summary:", summary)
print("Verdict:", verdict)